# Phase 1 (GitHub + Read + Q&A)

In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
from agents.mcp import MCPServerStreamableHttp, MCPServerStdio
from agents import Agent, trace, Runner, function_tool
from IPython.display import display, Markdown
import json
import subprocess
from openai import OpenAI

In [2]:
load_dotenv(override=True)

openai = OpenAI()

GITHUB_PAT = os.getenv("GITHUB_PAT")

### Defining MCP Servers

In [3]:
NB_MCP_MEMORY_PATH = Path.cwd() / 'memory.jsonl'

github_params = {"url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {GITHUB_PAT}"
        }
    }

memory_params = {"command": "npx","args": ["-y", "@modelcontextprotocol/server-memory"],"env": {"MEMORY_FILE_PATH": str(NB_MCP_MEMORY_PATH)}}

In [4]:
github_mcp_server = MCPServerStreamableHttp(params=github_params, client_session_timeout_seconds=30)
memory_mcp_server = MCPServerStdio(params=memory_params, client_session_timeout_seconds=30)

await github_mcp_server.connect()
await memory_mcp_server.connect()

### Cloning Tool

In [5]:
@function_tool
def clone_repo(repo_url: str, branch: str = None):
    """
    Clone a GitHub repository locally.

    :param repo_url: HTTPS or SSH repo URL
    :param clone_dir: Directory where repo will be cloned
    :param branch: Optional branch name
    """

    # Hardcoded path of cloning
    clone_dir = "./coding_agent/repos"

    os.makedirs(clone_dir, exist_ok=True)

    repo_name = repo_url.split("/")[-1].replace(".git", "")
    target_path = os.path.join(clone_dir, repo_name)

    if os.path.exists(target_path):
        print(f"Repo already exists at {target_path}")
        return target_path

    cmd = ["git", "clone"]

    if branch:
        cmd += ["-b", branch]

    cmd += [repo_url, target_path]

    try:
        subprocess.run(cmd, check=True)
        print(f"Cloned into {target_path}")
        return target_path
    except subprocess.CalledProcessError as e:
        print("Error cloning repo:", e)
        return None

### Building Index Tool

##### Summary Builder Function

In [6]:
def summary_builder(index_path: str) -> str:
	"""Creates summary of files of repository.

	Args:
		index_path: Path of index.

	Returns:
		Status
	"""
	try:
		print(f"Building summaries in Index . . . ")

		with open(index_path, 'r') as repo:
			json_repo = repo.read()
			list_repo = json.loads(json_repo)
		
		for file in list_repo:
			if file["summary"] is None:
				with open(file['file_path'], 'r', encoding='latin-1') as code:
					snippit = code.read()[:2000]
					summary = openai.chat.completions.create(
						model="gpt-4.1-nano",
						messages=[
							{"role": "system", "content": "Summarize the purpose of this code file briefly."},
							{"role": "user", "content": snippit}
							],
							)
			
				file["summary"] = summary.choices[0].message.content

		with open(index_path, "w") as f:
			json.dump(list_repo, f, indent=2)
		
		return 'success'

	except Exception as e:
		return f"Failed with an error {str(e)}"


In [7]:
@function_tool
def index_builder(repo_name: str) -> str:
	"""Builds index of repositroy in JSON format, which cointains 
		- file_name
		- file_path
		- summary

	Args:
		repo_name: The name of repository.

	Returns:
		It creates the index.
	"""
	index_path = Path.cwd() / 'coding_agent' / f"index_of_{repo_name}.json"

	if not index_path.exists():

		print("Building Index")

		index = []

		repo_path = Path.cwd() / "coding_agent" / "repos" / repo_name
		
		if repo_path.exists():
			files = list(repo_path.rglob("*"))
			
			# This removed all empty folders and .git files
			relevent_files = [i for i in files if i.suffix != '' and '.git' not in i.parts]

			for file in relevent_files:
				index.append({
					'name': file.name,
					'file_path': str(file),
					'summary': None
					})

			with open(Path.cwd() / 'coding_agent' / f"index_of_{repo_name}.json", "w") as file:
				json.dump(index, file, indent=2)
			
			print("Index built")

			summary_status = summary_builder(Path.cwd() / 'coding_agent' / f"index_of_{repo_name}.json")

			return {
				"index_status": "success",
				"summary_status": summary_status
			}
		else:
			return "This repo doesn't exists"
	
	else:
		return "Index already exists"

### Defining Agents

In [8]:
github_instructions = "You are smart coding agent which helps users to work on code." 

memory_instructions = "Use your memory tools to store the information about the repository. This has to be used everytime, whenever we talks about repo."

cloning_instructions = f"""If user wants to do some analysis or Q&A and wants you to work on the code, bugs or some feature request.
Then you should always tell user that first you have to clone the repo, and ask his permission for cloning. And if you have already cloned it, then you should not call this tool again and again. 

Below are the instructions for cloning the repo,
Always call 'clone_repo' tool. You can confirm this from user if he wants to clone it. 
In this you have to pass 2 arguments, out of which one is optional
1. URL of the repo: You need to pass full URL of the repo (This can be made "https://github.com/{{username}}/{{repo_name}}))
2. Branch (Optional): If user specifies the branch name then paste there, otherwise you can leave it.
"""

next_steps_instructions = """
Once cloned, you immedietly have to make an index of the repo by calling 'index_builder' tool, this is a MUST step. 
This tool takes only one argument, and that is repo name as it is. Be very careful on this. 
It will create a JSON file. And it'll store, file name, file path, and the description of each file of the repo.
And remember, if you have already built index of this same repo, then do not call this tool again and again.

Now whenver user asks you to work on any file or bug or asks you anything, you simply have to look in this index to determine which file you need to read, from there you can fetch the path.
"""

instructions = f"""{github_instructions}

{memory_instructions}

{cloning_instructions}
"""

In [9]:
github_agent = Agent (
	name="Github agent",
	instructions=instructions,
	model="gpt-4.1-mini",
	tools=[clone_repo, index_builder],
	mcp_servers=[github_mcp_server, memory_mcp_server]
)

# with trace("Github test"):
# 	result = await Runner.run(github_agent, "Can you clone my recruiters_email_reply_agent repo")

# display(Markdown(result.final_output))

### Implementing Memory

In [10]:
MEMORY_FILE = str(Path.cwd() / "chat_memory.json")

def load_memory():
    if not os.path.exists(MEMORY_FILE):
        return {
            "chat_history": [],
            # "repo_context": {}
        }
    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

In [11]:
def build_context(memory, user_input):
    chat_history = memory["chat_history"][-5:]

    context = ""

    for msg in chat_history:
        context += f"{msg['role']}: {msg['content']}\n"

    # Add repo ONLY when needed
   #  if intent in ["repo_query", "repo_task"]:
      #   context += f"\nRepo Context:\n{memory['repo_context']}\n"

    context += f"\nUser: {user_input}"

    return context

## Running our Agent

In [12]:
memory = load_memory()
c = 0

while True:
	user_input = input("User: ")

	display(Markdown(f"**User:** {user_input}"))

	context = build_context(memory, user_input)

	with trace("Coding agent"):
		result = await Runner.run(github_agent, context)

	display(Markdown(f"**Agent:** {result.final_output}"))

	memory["chat_history"].append({"role": "user", "content": result.input})

	memory["chat_history"].append({"role": "assistant", "content": result.final_output})

	# Trim history
	memory["chat_history"] = memory["chat_history"][-10:]

	save_memory(memory)

	c = c + 1

	if c == 20:
		break

**User:** Hello babe

**Agent:** Hello! How can I assist you today?

**User:** Can you help me with a repo of mine that is recruiters email reply one

**Agent:** Sure! To assist you effectively with your "recruiters email reply" repository, I'll need to clone the repo first. Could you please provide me with the URL of your repository? Also, if you want me to work on a specific branch, please let me know the branch name.

**User:** https://github.com/ManuSinghYadav/recruiters_email_reply_agent

...

**User:** First list the files with their description

Repo already exists at ./coding_agent/repos/recruiters_email_reply_agent
Building Index
Index built
Building summaries in Index . . . 


**Agent:** The repository "recruiters_email_reply_agent" has been cloned and indexed successfully. Here is the list of files with their descriptions from the repository:

Please let me know if you want me to list the files and their descriptions in detail.

**User:** 

CancelledError: 